# 2. Train and evaluate Neurotransformer

Trains the model with the hyperparameters in `configs/<dataset>.yaml` (the paper's settings), keeps the
checkpoint with the lowest validation loss (early stopping, patience 10), then reports accuracy, precision, recall, F1, specificity and AUROC. The same pipeline is
available from the command line as `python scripts/train.py --config configs/<dataset>.yaml`.

In [ ]:
from pathlib import Path

# Run from the repository root or from notebooks/.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATASET = "bonn"            # "nmt", "bonn" or "tuev"
MODEL = "neurotransformer"  # or an ablation: wo_res, wo_dil, wo_conv, wo_encoder, wo_decoder, wo_trans
EPOCHS = None               # None = use the config value
DATA_ROOT = None            # None = REPO / <data_root from config>
OUTPUT_DIR = REPO / "runs" / DATASET / MODEL

In [ ]:
import torch

from neurotransformer.data import build_dataloaders, get_spec
from neurotransformer.engine import append_results_csv, evaluate, fit, format_metrics, predict
from neurotransformer.models import build_model
from neurotransformer.utils import (build_criterion, build_optimizer, build_scheduler, count_params,
                                    load_config, resolve_class_counts)

cfg = load_config(REPO / "configs" / f"{DATASET}.yaml")
cfg["model"] = MODEL
if EPOCHS is not None:
    cfg["epochs"] = EPOCHS
data_root = Path(DATA_ROOT) if DATA_ROOT else REPO / cfg["data_root"]
spec = get_spec(DATASET)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg

In [ ]:
loaders = build_dataloaders(str(data_root), DATASET, cfg["batch_size"], cfg["num_workers"])
print({split: len(loader.dataset) for split, loader in loaders.items()})

model = build_model(MODEL, spec.n_classes).to(device)
cfg["loss"] = resolve_class_counts(cfg.get("loss"), loaders["train"].dataset)  # inverse-frequency class weights
criterion = build_criterion(cfg["loss"], device)
print(f"{MODEL}: {count_params(model):,} parameters on {device}")

## Train

In [ ]:
optimizer = build_optimizer(model.parameters(), cfg["optimizer"])
scheduler = build_scheduler(optimizer, cfg.get("scheduler"))
checkpoint = OUTPUT_DIR / "best.pth"

history = fit(model, loaders["train"], loaders["eval"], optimizer, criterion, device, cfg["epochs"],
              spec.n_classes, str(checkpoint), scheduler, cfg.get("monitor", "eval_loss"), cfg.get("patience"))
history.to_csv(OUTPUT_DIR / "history.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history["epoch"], history["train_loss"], label="train")
ax1.plot(history["epoch"], history["eval_loss"], label="eval")
ax1.set_title("Loss")
ax2.plot(history["epoch"], history["train_acc"], label="train")
ax2.plot(history["epoch"], history["eval_acc"], label="eval")
ax2.set_title("Accuracy")
for ax in (ax1, ax2):
    ax.set_xlabel("Epoch")
    ax.legend()
plt.tight_layout()

## Evaluate the best checkpoint

To evaluate an existing checkpoint without training, run the setup cells above, skip **Train**, and set
`checkpoint` to your `.pth` file.

In [ ]:
from neurotransformer.visualization import plot_confusion_matrix, plot_roc_curves

model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))
splits = [s for s in ("eval", "test") if s in loaders]

fig, axes = plt.subplots(2, len(splits), figsize=(6 * len(splits), 10), squeeze=False)
for col, split in enumerate(splits):
    metrics, cm = evaluate(model, loaders[split], device, spec.n_classes, criterion)
    append_results_csv(str(OUTPUT_DIR / f"{split}_results.csv"), str(checkpoint), metrics, cm)
    print(f"[{split}] {format_metrics(metrics)}")

    labels, _, probs, _ = predict(model, loaders[split], device)
    plot_confusion_matrix(cm, spec.class_names, ax=axes[0, col], title=f"{DATASET} - {split}")
    plot_roc_curves(labels, probs, spec.class_names, ax=axes[1, col], title=f"ROC - {split}")
plt.tight_layout()